# Bipolarity metrics pilot: 3 approaches compared on 20 articles

Tests three simple, automated, CPU-only proxies for how much a news article frames an issue as strictly two opposing sides (see `src/bipolarity.py` for full docstrings):

1. **`dichotomy_marker_density`** -- keyword/pattern density of explicit binary-framing language (`either...or`, `vs`, `two sides`, ...). Cheapest, most transparent; only catches explicit phrasing.
2. **`entity_sentiment_gap`** -- sentiment-analysis proxy for *implicit* bipolar framing: finds the two most-mentioned named entities, scores sentiment of sentences mentioning each (VADER), reports the gap between them.
3. **`antonym_cooccurrence_density`** -- density of WordNet antonym pairs both appearing in the article (e.g. "safe"/"dangerous"), per 1000 words.

None of these are validated against human-labeled bipolarity judgments -- they're cheap, transparent proxies for comparison, not a claimed ground truth. This notebook runs all three on the first 20 articles of `australia_498sample_climatechange.csv` and compares them.

No LLM, no GPU needed -- any Colab runtime works.

## 1. Install dependencies

In [ ]:
!pip install -q spacy nltk pandas
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("vader_lexicon", quiet=True)

## 2. Clone this repo (branch: `claude/bipolarity-metrics`)

In [ ]:
import os

REPO_URL = "https://github.com/hrauxloh/DAAD_Destructive_polarization"
BRANCH = "claude/bipolarity-metrics"
REPO_DIR = "/content/DAAD_Destructive_polarization"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

## 3. Run all three metrics on the first 20 articles

In [ ]:
import csv
import pandas as pd
from src.bipolarity import compute_bipolarity_table

with open("australia_498sample_climatechange.csv", newline="", encoding="utf-8") as f:
    articles = list(csv.DictReader(f))[:20]

print(f"testing on {len(articles)} articles")
rows = compute_bipolarity_table(articles)
bipolarity_df = pd.DataFrame(rows)
bipolarity_df.to_csv("aus_bipolarity_pilot.csv", index=False)
bipolarity_df

## 4. Compare the three approaches

Correlation between the three main scores -- do they agree on which articles are "most bipolar," or are they picking up on different things?

In [ ]:
compare_cols = ["dichotomy_marker_density", "entity_sentiment_gap", "antonym_cooccurrence_density"]
print(bipolarity_df[compare_cols].corr())
bipolarity_df[["document_id", "title"] + compare_cols]

## 5. Rank comparison
For each metric, which articles rank highest? Overlap (or lack of it) across the three rankings tells you whether they're measuring the same thing or genuinely different aspects of "bipolarity."

In [ ]:
for col in compare_cols:
    print(f"\n=== Top 5 by {col} ===")
    top5 = bipolarity_df.nlargest(5, col)[["document_id", "title", col]]
    for _, row in top5.iterrows():
        print(f"  [{row[\'document_id\']}] {row[col]:.4f} -- {row[\'title\'][:70]}")

## 6. Download the results

In [ ]:
from google.colab import files
files.download("aus_bipolarity_pilot.csv")

## Notes / limitations
- **`dichotomy_marker_density` was 0 for all 20 articles in initial local testing** -- explicit binary phrasing ("either...or", "vs", "two sides") is rare in this climate-news corpus. That's an informative finding about the metric's known limitation (it only catches *explicit* framing), not a bug -- if this holds at scale, this metric alone won't be useful for this corpus and `entity_sentiment_gap` or `antonym_cooccurrence_density` are more likely to show real variation.
- `entity_sentiment_gap` requires at least 2 distinct named entities to be detected in the article; articles with fewer than 2 will show `None` for this metric.
- `antonym_cooccurrence_density` uses WordNet antonym relations without word-sense disambiguation, so it can occasionally surface a spurious pair (e.g. an antonym relation tied to an uncommon word sense that doesn't match how the word is actually used in the article) -- check `antonym_pairs_sample` in the output before trusting a high score.
- None of these three has been validated against human bipolarity judgments in this project -- treat them as exploratory signals to compare against each other and against the LLM-based `black_and_white` detection (`notebooks/colab_propaganda_poc.ipynb`), not as ground truth on their own.